# RSMLM — LLM with no KV cache, trained on A100

**Architecture recap**
- Causal self-attention over the **current chunk only** (fixed `block_size`) — no growing KV cache
- Each layer also does **RSM cross-attention** into 512 fixed memory slots
- After each chunk, the model's hidden state is written back to RSM (EMA)
- RSM resets at document boundaries

**Runtime:** Set `Runtime → Change runtime type → A100 GPU` before running.

**Secret required:** Add a Colab secret named `gitRSM` containing your GitHub PAT with read access to `vaibhavsadgir50/RSMLM`.

In [ ]:
# ── CELL 0 — GPU check ────────────────────────────────────────────────────
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU — switch runtime to A100')

In [ ]:
# ── CELL 1 — Clone repo from private GitHub using Colab secret ────────────
import os, sys, shutil, subprocess
from google.colab import userdata

TOKEN = userdata.get('gitRSM')          # Colab secret name
if not TOKEN:
    raise RuntimeError(
        'Secret gitRSM not found. '
        'Go to Colab sidebar → Secrets → add key=gitRSM, value=<your GitHub PAT>'
    )

REPO_DIR = '/content/RSMLM'
REPO_URL = f'https://{TOKEN}@github.com/vaibhavsadgir50/RSMLM.git'

if os.path.isdir(os.path.join(REPO_DIR, 'lm')):
    print('Repo already present — pulling latest')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    rc = subprocess.run(
        ['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR],
        capture_output=True, text=True
    )
    if rc.returncode != 0:
        print('STDERR:', rc.stderr)
        raise RuntimeError('Clone failed — check gitRSM secret has repo read access')

sys.path.insert(0, REPO_DIR)
print('Repo ready at', REPO_DIR)
print('Files:', os.listdir(REPO_DIR))

In [ ]:
# ── CELL 2 — Mount Drive for checkpoints ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT = '/content/drive/MyDrive/RSMLM/checkpoints/'
DATA_DIR   = '/content/drive/MyDrive/RSMLM/data/'
os.makedirs(DRIVE_CKPT, exist_ok=True)
os.makedirs(DATA_DIR,   exist_ok=True)
print('Drive checkpoint dir:', DRIVE_CKPT)

## Phase 0 — Train RSM on text (run this BEFORE LM training)

RSM pre-training gives the memory meaningful structure before the LM starts writing to it.

**Stage 1** — trains `EncoderProjection` via InfoNCE so that related text chunks cluster together in embedding space.  
**Stage 2** — trains `query_proj` inside `RSM.read()` so the RSM learns to retrieve what the query is asking for.

Run this section once. Checkpoints land in `DRIVE_CKPT/rsm_s1_proj.pt` and `DRIVE_CKPT/rsm_s2_retrieval.pt`.  
The LM training cell below will load them automatically.

In [ ]:
# ── RSM PRE-TRAIN — Hyperparameters ──────────────────────────────────────
RSM_D_MODEL    = 512         # encoder projection output dim (must match LMConfig.rsm_d_model)
RSM_N_SLOTS    = 512         # RSM slot count
RSM_ENCODER_DIM = 768        # FallbackEncoder output dim (always 768)
RSM_CHUNK_SIZE = 128         # words per chunk
RSM_OVERLAP    = 32          # word overlap between chunks

# Stage 1 — projection training (InfoNCE)
RSM_S1_EPOCHS  = 15
RSM_S1_LR      = 3e-4
RSM_S1_BATCH   = 64

# Stage 2 — retrieval training (query_proj + projection)
RSM_S2_EPOCHS  = 20
RSM_S2_LR      = 1e-4
RSM_S2_BATCH   = 16

# Data source — any of:
#   'shakespeare'  : auto-download TinyShakespeare
#   'hf:wikitext'  : HuggingFace wikitext (requires: pip install datasets)
#   '/path/to/file.txt' or '/path/to/dir/'
RSM_SOURCE     = 'shakespeare'
RSM_HF_SUBSET  = 'wikitext-2-raw-v1'   # only used when RSM_SOURCE = 'hf:wikitext'

RSM_CKPT_DIR   = os.path.join(DRIVE_CKPT, 'rsm/')
os.makedirs(RSM_CKPT_DIR, exist_ok=True)
print(f'RSM checkpoints will land in: {RSM_CKPT_DIR}')

In [ ]:
# ── RSM PRE-TRAIN — Stage 1: Projection (InfoNCE) ────────────────────────
# Trains EncoderProjection so same-doc chunks cluster, cross-doc chunks separate.
# Gradient path: loss → projection weights
# RSM is not involved in Stage 1.

import sys
sys.path.insert(0, REPO_DIR)

import importlib
import training.train_rsm as _trsm; importlib.reload(_trsm)

from training.train_rsm import stage1_projection, _load_source
from rsm_data.pairs import make_contrastive_pairs, pair_batches
from encoder.nomic_encoder import NomicTextEncoder
from encoder.projection import EncoderProjection
from config import get_config

rsm_cfg = get_config({
    'd_model'    : RSM_D_MODEL,
    'n_slots'    : RSM_N_SLOTS,
    'encoder_dim': RSM_ENCODER_DIM,
})

rsm_encoder    = NomicTextEncoder(RSM_ENCODER_DIM, use_transformers=False).to(device)
rsm_projection = EncoderProjection(RSM_ENCODER_DIM, RSM_D_MODEL).to(device)
rsm_encoder.eval()

print('Loading documents for RSM training...')
all_docs = _load_source(
    RSM_SOURCE, RSM_CHUNK_SIZE, RSM_OVERLAP,
    RSM_HF_SUBSET, 'text', 'train', max_docs=None,
)
from rsm_data.loader import TextLoader
s = TextLoader.stats(all_docs)
print(f'  {s["n_docs"]:,} docs | {s["n_chunks"]:,} chunks | {s["total_words"]:,.0f} words')

val_n       = max(1, int(len(all_docs) * 0.1))
train_docs  = all_docs[val_n:]
val_docs    = all_docs[:val_n]
train_pairs = make_contrastive_pairs(train_docs, max_span=4, n_neg=7, max_pairs_per_doc=25)
val_pairs   = make_contrastive_pairs(val_docs,   max_span=4, n_neg=7, max_pairs_per_doc=10)
print(f'  train pairs: {len(train_pairs):,} | val pairs: {len(val_pairs):,}')

S1_CKPT = os.path.join(RSM_CKPT_DIR, 'rsm_s1_proj.pt')
stage1_projection(
    rsm_encoder, rsm_projection, train_pairs,
    epochs=RSM_S1_EPOCHS, batch_size=RSM_S1_BATCH, lr=RSM_S1_LR,
    device=device, val_pairs=val_pairs,
    checkpoint_path=S1_CKPT,
)
print(f'Stage 1 complete. Checkpoint: {S1_CKPT}')

In [ ]:
# ── RSM PRE-TRAIN — Stage 2: Retrieval (query_proj + projection) ─────────
# Trains RSM.query_proj so the memory learns to retrieve what each query is looking for.
# Also fine-tunes projection jointly.
# Gradient path: loss → query_proj params + projection params

from training.train_rsm import stage2_retrieval
from rsm_data.pairs import make_retrieval_triples

train_triples = make_retrieval_triples(train_docs, split_frac=0.6, n_queries_per_doc=4)
val_triples   = make_retrieval_triples(val_docs,   split_frac=0.6, n_queries_per_doc=4)
print(f'train triples: {len(train_triples):,} | val triples: {len(val_triples):,}')

S2_CKPT = os.path.join(RSM_CKPT_DIR, 'rsm_s2_retrieval.pt')
stage2_retrieval(
    rsm_encoder, rsm_projection, rsm_cfg, train_triples,
    epochs=RSM_S2_EPOCHS, batch_size=RSM_S2_BATCH, lr=RSM_S2_LR,
    device=device, val_triples=val_triples,
    checkpoint_path=S2_CKPT, s1_checkpoint=S1_CKPT,
)
print(f'Stage 2 complete. Checkpoint: {S2_CKPT}')

# Quick retrieval eval
from training.eval_rsm import eval_retrieval
scores = eval_retrieval(rsm_encoder, rsm_projection, rsm_cfg, val_docs,
                        n_docs=min(100, len(val_docs)), device=device)
print('\nPost-Stage-2 retrieval metrics:')
for k, v in scores.items():
    label = k if k == 'n_queries' else k
    print(f'  {label:<14}: {int(v) if k == "n_queries" else f"{v:.4f"}')


---
## Phase 1 — Train the LM (uses pre-trained RSM)

The cells below train `RSMLM`. If you ran Phase 0, the LM's write path uses an already-structured RSM space.  
Cell 7 (Resume checkpoint) will also load the trained `query_proj` from Phase 0 if available.

In [ ]:
# ── CELL 3 — Hyperparameters (edit here) ─────────────────────────────────

# Model preset: 'tiny' | 'small' | 'base'
#   tiny  — 0.5M params, CPU fast
#   small — 3M  params, GPU ~5 min / 5k steps
#   base  — 15M params, A100 ~15 min / 10k steps
PRESET      = 'base'

STEPS       = 20_000          # total optimizer steps
LR          = 3e-4            # peak learning rate (cosine schedule)
GRAD_CLIP   = 1.0
LOG_EVERY   = 200
SAVE_EVERY  = 1_000

# Override individual dims (set to None to use preset defaults)
D_MODEL     = None            # e.g. 512
N_LAYER     = None            # e.g. 6
N_HEAD      = None            # e.g. 8
BLOCK_SIZE  = None            # tokens per chunk; no KV cache beyond this
RSM_SLOTS   = None            # number of RSM memory slots

DATA_PATH   = os.path.join(DATA_DIR, 'shakespeare.txt')
DEVICE      = 'cuda'

print(f'Preset: {PRESET} | Steps: {STEPS} | Device: {DEVICE}')

In [ ]:
# ── CELL 4 — Imports ─────────────────────────────────────────────────────
import importlib
import math
import time
import urllib.request

import torch
import torch.nn.functional as F

# Reload modules so re-running cells picks up any edits
import config as _cfg_mod;       importlib.reload(_cfg_mod)
import tensor_utils as _tu_mod;  importlib.reload(_tu_mod)
import rsm.core as _rc;          importlib.reload(_rc)
import lm.model as _lm_mod;      importlib.reload(_lm_mod)
import training.train_lm as _tr; importlib.reload(_tr)

from config import get_config
from lm.model import RSMLM, LMConfig
from rsm.core import ResonantStateMemory
from training.train_lm import (
    load_text, build_vocab, encode, make_chunks,
    chunk_importance, evaluate, generate_sample
)
from training.checkpoint import save_checkpoint, load_checkpoint

device = torch.device(DEVICE)
print('PyTorch:', torch.__version__, '| Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── CELL 5 — Load dataset ─────────────────────────────────────────────────
text = load_text(DATA_PATH)
chars, stoi, itos = build_vocab(text)
print(f'Dataset: {len(text):,} chars | vocab size: {len(chars)}')
print('Sample:', repr(text[:100]))

In [ ]:
# ── CELL 6 — Build model + RSM ────────────────────────────────────────────
preset_fn = {'tiny': LMConfig.tiny, 'small': LMConfig.small, 'base': LMConfig.base}
cfg = preset_fn[PRESET](vocab_size=len(chars))

# Apply overrides
if D_MODEL:    cfg.d_model    = D_MODEL
if N_LAYER:    cfg.n_layer    = N_LAYER
if N_HEAD:     cfg.n_head     = N_HEAD
if BLOCK_SIZE: cfg.block_size = BLOCK_SIZE
if RSM_SLOTS:
    cfg.rsm_n_slots = RSM_SLOTS
    cfg.rsm_d_model = cfg.d_model

model = RSMLM(cfg).to(device)
rsm   = model.build_rsm(device)

n_params = model.count_parameters()
print(f'\nConfig: {cfg}')
print(f'\nTotal parameters: {n_params:,}')
print('Breakdown:')
for k, v in model.parameter_breakdown().items():
    print(f'  {k:<14}: {v:>10,}')

# VRAM estimate
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        dummy = torch.randint(0, cfg.vocab_size, (4, cfg.block_size), device=device)
        _ = model(dummy, rsm)
    print(f'\nEstimated VRAM (batch=4): {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

In [ ]:
# ── CELL 7 — (Optional) Resume from checkpoint ────────────────────────
RESUME    = True        # set False to train from scratch
CKPT_PATH = os.path.join(DRIVE_CKPT, 'rsmlm_best.pt')

# Stage 2 RSM checkpoint — load query_proj trained in Phase 0 if present
RSM_S2_CKPT = os.path.join(DRIVE_CKPT, 'rsm/', 'rsm_s2_retrieval.pt')

start_step = 0
optimizer  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.1, betas=(0.9, 0.95))

if RESUME and os.path.exists(CKPT_PATH):
    ck = load_checkpoint(CKPT_PATH)
    model.load_state_dict(ck['model_state_dict']['model'])
    if ck.get('optimizer_state_dict'):
        optimizer.load_state_dict(ck['optimizer_state_dict'])
    start_step = int(ck.get('epoch', 0)) + 1
    print(f'Resumed from step {start_step} | val_loss={ck["val_loss"]:.4f}')
    if 'rsm' in ck['model_state_dict']:
        rsm.summaries.copy_(ck['model_state_dict']['rsm'].to(device))
        print('RSM state restored from LM checkpoint')
else:
    print('Training from scratch')

# Load trained query_proj from Phase 0 (RSM pre-training Stage 2)
if os.path.exists(RSM_S2_CKPT):
    s2_ck = load_checkpoint(RSM_S2_CKPT)
    rsm.query_proj.load_state_dict(s2_ck['model_state_dict']['query_proj'])
    print(f'Loaded pre-trained query_proj from {RSM_S2_CKPT}')
    print('  RSM will start with a meaningful read projection instead of identity')
else:
    print('No Stage 2 RSM checkpoint found — query_proj starts as identity')
    print(f'  (expected at {RSM_S2_CKPT} — run Phase 0 first for best results)')

In [ ]:
# ── CELL 8 — Prepare data chunks ─────────────────────────────────────────
tokens = encode(text, stoi)
split  = int(0.9 * len(tokens))
train_tok, val_tok = tokens[:split], tokens[split:]
train_chunks = make_chunks(train_tok, cfg.block_size, device)
val_chunks   = make_chunks(val_tok,   cfg.block_size, device)

print(f'Train chunks: {len(train_chunks):,} × {cfg.block_size} tokens')
print(f'Val   chunks: {len(val_chunks):,} × {cfg.block_size} tokens')

In [ ]:
# ── CELL 9 — Training loop ────────────────────────────────────────────────
# Uses AMP (mixed precision) on A100 for ~2x speed.
# RSM cross-attention gives the model long-range memory without any KV cache.

USE_AMP = (device.type == 'cuda')
scaler  = torch.cuda.amp.GradScaler(enabled=USE_AMP)

WARMUP_STEPS  = min(500, STEPS // 10)
doc_reset_every = max(1, len(train_chunks) // 10)

def get_lr(step: int) -> float:
    if step < WARMUP_STEPS:
        return LR * step / max(WARMUP_STEPS, 1)
    t = (step - WARMUP_STEPS) / max(STEPS - WARMUP_STEPS, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * t))

best_val_loss = float('inf')
chunk_idx     = 0
t0 = time.time()
model.train()

# Metrics for plotting
train_losses, val_losses, steps_log = [], [], []

for step in range(start_step, STEPS):
    # LR
    new_lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg['lr'] = new_lr

    # RSM reset at document boundary
    if chunk_idx % doc_reset_every == 0:
        rsm.summaries.fill_(0.0)
        torch.nn.init.orthogonal_(rsm.summaries)

    x, y = train_chunks[chunk_idx % len(train_chunks)]
    chunk_idx += 1

    # Forward + loss (AMP)
    with torch.cuda.amp.autocast(enabled=USE_AMP):
        logits, hidden = model(x.unsqueeze(0), rsm)
        loss = F.cross_entropy(logits.view(-1, cfg.vocab_size), y.view(-1))

    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()

    # RSM write-back (no gradient)
    model.write_to_rsm(hidden, rsm, importance=chunk_importance(x, itos))

    if (step + 1) % LOG_EVERY == 0 or step == start_step:
        elapsed = time.time() - t0
        print(
            f'step {step+1:6d}/{STEPS} | '
            f'loss {loss.item():.4f} | '
            f'lr {new_lr:.2e} | '
            f'{elapsed:.1f}s'
        )
        train_losses.append(loss.item())
        steps_log.append(step + 1)
        t0 = time.time()

    if (step + 1) % SAVE_EVERY == 0:
        val_loss = evaluate(model, rsm, val_chunks, cfg, device, n_chunks=100)
        val_losses.append(val_loss)
        print(f'  → val_loss {val_loss:.4f} | best {best_val_loss:.4f}')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(
                CKPT_PATH,
                {'model': model.state_dict(), 'rsm': rsm.summaries.cpu()},
                optimizer.state_dict(),
                step,
                val_loss,
                {'lm_config': cfg.__dict__},
            )
            print(f'  → checkpoint saved to Drive')
        model.train()

print(f'\nTraining done. Best val_loss: {best_val_loss:.4f}')

In [ ]:
# ── CELL 10 — Plot loss curves ────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(steps_log, train_losses, label='train loss', alpha=0.7)
if val_losses:
    val_x = [SAVE_EVERY * (i + 1) for i in range(len(val_losses))]
    ax.plot(val_x, val_losses, 'o-', label='val loss', linewidth=2)
ax.set_xlabel('step')
ax.set_ylabel('cross-entropy loss')
ax.set_title('RSMLM training — no KV cache')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_CKPT, 'loss_curve.png'), dpi=150)
plt.show()
print('Loss curve saved to Drive')

In [ ]:
# ── CELL 11 — Generate samples ────────────────────────────────────────────
PROMPTS = [
    '\n',
    'HAMLET:\n',
    'To be, or not to be',
    'The king said',
]
N_TOKENS    = 300
TEMPERATURE = 0.8
TOP_K       = 40

for prompt in PROMPTS:
    sample = generate_sample(
        model, rsm, itos, stoi, device, cfg,
        prompt=prompt, n=N_TOKENS,
        temperature=TEMPERATURE, top_k=TOP_K
    )
    print(f'--- prompt: {repr(prompt)} ---')
    print(sample)
    print()

In [ ]:
# ── CELL 12 — RSM diagnostics ────────────────────────────────────────────
# Shows how well the RSM slots are being used.

import torch

with torch.no_grad():
    norms = rsm.summaries.norm(dim=-1).cpu()

print('RSM slot diagnostics')
print(f'  n_slots  : {rsm.n_slots}')
print(f'  d_model  : {rsm.d_model}')
print(f'  norm mean: {norms.mean():.4f}')
print(f'  norm std : {norms.std():.4f}')
print(f'  norm min : {norms.min():.4f}')
print(f'  norm max : {norms.max():.4f}')

# Slot utilization: fraction of slots with norm > 1% of mean
threshold = norms.mean() * 0.01
utilization = (norms > threshold).float().mean().item()
print(f'  utilization: {utilization*100:.1f}% of slots active')

# Slot similarity distribution (how diverse are slots?)
from tensor_utils import l2_normalize
s = l2_normalize(rsm.summaries.cpu(), dim=-1)    # (n_slots, d)
gram = s @ s.T                                    # (n_slots, n_slots)
off_diag = gram[~torch.eye(rsm.n_slots, dtype=torch.bool)]
print(f'  inter-slot cosine: mean={off_diag.mean():.3f} std={off_diag.std():.3f}')
print('  (low mean = diverse slots = good memory coverage)')

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(norms.numpy(), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('RSM slot norms')
axes[0].set_xlabel('L2 norm')
axes[1].hist(off_diag.numpy(), bins=50, color='coral', edgecolor='white')
axes[1].set_title('Inter-slot cosine similarity')
axes[1].set_xlabel('cosine')
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_CKPT, 'rsm_diagnostics.png'), dpi=150)
plt.show()

In [ ]:
# ── CELL 13 — Ablation: RSM vs no-RSM (zero slots) ───────────────────────
# Measures val loss when RSM slots are zeroed out.
# A large gap confirms RSM is doing real work.

# Evaluate with live RSM
val_with_rsm = evaluate(model, rsm, val_chunks, cfg, device, n_chunks=100)

# Evaluate with blank RSM
blank_rsm = model.build_rsm(device)
blank_rsm.summaries.zero_()
val_no_rsm = evaluate(model, blank_rsm, val_chunks, cfg, device, n_chunks=100)

print(f'Val loss  WITH RSM: {val_with_rsm:.4f}')
print(f'Val loss   NO RSM : {val_no_rsm:.4f}')
print(f'RSM benefit       : {val_no_rsm - val_with_rsm:.4f} nats')
print()
if val_with_rsm < val_no_rsm:
    print('✓ RSM is providing useful context — model learned to read it')
else:
    print('RSM benefit not yet visible — train longer or increase RSM slots')